# [8] Simple Approach

## Imports

In [1]:
import env

Environment setup complete. Project path: C:\Users\BREW\Desktop\AI_Study\epistemic-detection


In [2]:
from epidec.datasets import BalancedBaggedSWUnivDaconDataset, SWUnivDaconDataset

from transformers import ElectraPreTrainedModel, ElectraModel, ElectraConfig, AutoTokenizer
from torch.utils.data import DataLoader
from torch import nn, optim
import torch

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import gc
import re

In [3]:
from sklearn.exceptions import UndefinedMetricWarning
import warnings

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

In [4]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

Testing tqdm:   0%|          | 0/1000 [00:00<?, ?it/s]

In [5]:
# project name
PROJECT_NAME = "8_simple"

### Check GPU Availability

In [6]:
!nvidia-smi

Mon Jul 14 12:47:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   51C    P8              1W /   78W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# Set CUDA Device
device_num = 0

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

INFO: Using device - cuda:0


## Load Datasets

In [8]:
DATA_ROOT = "./data"

train_dataset = BalancedBaggedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1, balancing_ratio=2, bagging_size=5)
valid_dataset = BalancedBaggedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1, balancing_ratio=1)
valid_totalset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = BalancedBaggedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Valid Total - {len(valid_totalset)}, Test - {len(test_dataset)}")

INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Total 71950 datas are used, 8309 datas remain unused.
INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset 'swuniv_dacon' already exists at data. Skipping download.
INFO: Dataset loaded successfully. Train - 21585, Valid - 1600, Valid Total - 9718, Test - 1962


In [9]:
train_dataset[1]

[['파군(巴郡)은 중국의 고대 행정 구역이다. 현재의 충칭 시와 쓰촨성 서부 지역을 관할했던 군이다.\n기원전 316년, 진나라에서 파나라를 멸하고 그 자리에 처음으로 파군을 설치했다. 치소는 강주현이라고도 하는 강현(현재의 충칭 시 북)이다. 관할 구역은 대략 지금의 쓰촨성 랑중시 이동과 충칭시 푸링 이서의 사이다. 문헌과 출토자료에서 확인된 속현은 총 11현으로, 다음과 같다. 이외에 한중군이 12현, 촉군이 18현으로, 이는 《한서·고제기》에 “패공을 고쳐 한왕으로 삼고 파·촉·한중 41현에서 왕노릇하게 했다.”와 부합한다.\n기원전 206년 항우가 진나라를 무너뜨리고 중국 전역에 열여덟 제후왕을 세웠다. 이때 파군은 촉군과 한중군과 함께 한나라를 세우고 고제의 통치를 받게 되었다.\n신나라에서는 다음과 같은 현의 이름을 변경했다.\n초주의 《파기(巴記)》에 따르면 건안 6년(200년) 익주목 유장이 파군의 점강 이서를 분리하여 파서군으로 독립시켰다고 전해진다.\n촉한시기에는 파군의 동부지역이 파동군으로 나뉘었고 동남부지역은 부릉군으로 분리되었다.\n서진에서는 익주의 동부지역을 양주로 나누었으며 파군과 새로 설립된 파서군, 파동군 그리고 부릉군 모두 양주로 이관되었다.\n영가의 난시기 사천분지를 장악한 성한의 이웅에 의해 점령되면서 성한의 통치하에 놓였다. 성한이 멸망한 뒤 전진에 소속되었다가 동진에 의해 수복되면서 중국왕조의 통치하로 다시 들어가게 되었다.',
  1],
 ['파군(巴郡)은 중국의 고대 행정 구역이다. 현재의 충칭 시와 쓰촨성 서부 지역을 관할했던 군이다.\n기원전 316년, 진나라에서 파나라를 멸하고 그 자리에 처음으로 파군을 설치했다. 치소는 강주현이라고도 하는 강현(현재의 충칭 시 북)이다. 관할 구역은 대략 지금의 쓰촨성 랑중시 이동과 충칭시 푸링 이서의 사이다. 문헌과 출토자료에서 확인된 속현은 총 11현으로, 다음과 같다. 이외에 한중군이 12현, 촉군이 18현으로, 이는 《한서·고제기》에 “패공을 고쳐 한왕으로 삼고 파·촉

In [10]:
valid_dataset[0]

("샤쥔춘(沙俊春, 1984년 3월 17일~)은 중국 출신의 스타크래프트 프로게이머였습니다. SK텔레콤 T1 팀 소속으로 프로토스를 주로 사용했죠.\n2004년 루오시안과 같이 SK텔레콤 T1에 입단하였다.\n2007년 5월 한국에서의 선수 생활을 끝내고 중국으로 돌아갔습니다. 그렇게 그의 국내 무대는 막을 내렸습니다.\nWCG 국가대표로 출전해서 WCG 2007 그랜드 파이널 결승에선 대한민국의 송병구에게 패했지만 은메달을 따냈다.\n2009년 스웨덴과 독일의 이스포츠팀인 SK 게이밍에 입단하였다.\n2009년에는 중국에서 프로게임팀 소속으로 선수이자 감독을 맡았습니다.\n2010년 게임단이 해체된 후 은퇴했고요 지금은 개인 사업을 하고 있습니다.\nWCG2007 그랜드파이널 8강에서 한국의 프로게이머 마재윤을 2:1로 격파하면서 '샤본좌, 대륙의혁명가' 등의 별명을 얻었다. 반면 마재윤은 큰 비난을 받으며 마완용이라는 불명예스러운 별명까지 얻었다.\n한국 선수들과는 늘 끈질긴 악연이었죠. 쉽게 이길 수 없었던 상대였습니다. 그들의 힘과 투지는 정말 대단했어요. 한국 선수와의 경기는 늘 긴장의 연속이었고 승부를 예측할 수 없었습니다.\n샤쥔춘은 외국 프로게이머로서는 한국 프로게이머를 상당수 잡은 몇 안되는 프로게이머로도 유명하다.\n이 중에서 그간 잡은 프로게이머 목록이 최연성 마재윤 김정우 등 국내 유명 프로게이머 상당수일 정도.\n최근에는 IEF 2009에서 김정우에게 전진 2게이트를 선보이면서 승리를 챙기기도 했으며,\n이 때 팬들은 다시 그를 보고 샤본좌 라고 연호했고 김정우는 매완용, 매데요 시 라는 불명예스러운 별명을 얻고야 만다.\n김택용과의 끈질긴 인연.\n샤쥔춘은 마재윤과 김정우를 모두 꺾고 최연성까지 잡았지만 김택용만은 넘지 못했습니다. IEF 2008에서는 3대 0 완패를 당했고 IEF 2009에서도 김택용이 탈락하기 전까지는 김택용의 그림자에 가려져 있었습니다. 결국 WCG에서 설욕을 다짐했지만 김택용의 다크 템플러에 무너지고 말았습니다

#### Sentence segmentation

In [11]:
import re

def split_sentence(text):
    text = text.strip()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    pattern = r'([.!?]+)(\s+)(?=[A-Z가-힣])'

    sentences = []
    last_end = 0

    for match in re.finditer(pattern, text):
        sentence = text[last_end:match.end()-len(match.group(2))].strip()
        if sentence:
            sentences.append(sentence)
        last_end = match.end()-len(match.group(2))

    if last_end < len(text):
        last_sentence = text[last_end:].strip()
        if last_sentence:
            sentences.append(last_sentence)

    return sentences

In [12]:
# Train set
for i in range(len(train_dataset)):
    for j in range(train_dataset.bagging_size):
        train_dataset.data[j][i] = split_sentence(train_dataset.data[j][i])

# Valid set
valid_dataset.data = [split_sentence(i) for i in valid_dataset.data]

# Test set
test_dataset.data = [split_sentence(i) for i in test_dataset.data]

In [13]:
train_dataset[1]

[[['파군(巴郡)은 중국의 고대 행정 구역이다.',
   '현재의 충칭 시와 쓰촨성 서부 지역을 관할했던 군이다.',
   '기원전 316년, 진나라에서 파나라를 멸하고 그 자리에 처음으로 파군을 설치했다.',
   '치소는 강주현이라고도 하는 강현(현재의 충칭 시 북)이다.',
   '관할 구역은 대략 지금의 쓰촨성 랑중시 이동과 충칭시 푸링 이서의 사이다.',
   '문헌과 출토자료에서 확인된 속현은 총 11현으로, 다음과 같다.',
   '이외에 한중군이 12현, 촉군이 18현으로, 이는 《한서·고제기》에 “패공을 고쳐 한왕으로 삼고 파·촉·한중 41현에서 왕노릇하게 했다.”와 부합한다.',
   '기원전 206년 항우가 진나라를 무너뜨리고 중국 전역에 열여덟 제후왕을 세웠다.',
   '이때 파군은 촉군과 한중군과 함께 한나라를 세우고 고제의 통치를 받게 되었다.',
   '신나라에서는 다음과 같은 현의 이름을 변경했다.',
   '초주의 《파기(巴記)》에 따르면 건안 6년(200년) 익주목 유장이 파군의 점강 이서를 분리하여 파서군으로 독립시켰다고 전해진다.',
   '촉한시기에는 파군의 동부지역이 파동군으로 나뉘었고 동남부지역은 부릉군으로 분리되었다.',
   '서진에서는 익주의 동부지역을 양주로 나누었으며 파군과 새로 설립된 파서군, 파동군 그리고 부릉군 모두 양주로 이관되었다.',
   '영가의 난시기 사천분지를 장악한 성한의 이웅에 의해 점령되면서 성한의 통치하에 놓였다.',
   '성한이 멸망한 뒤 전진에 소속되었다가 동진에 의해 수복되면서 중국왕조의 통치하로 다시 들어가게 되었다.'],
  1],
 [['파군(巴郡)은 중국의 고대 행정 구역이다.',
   '현재의 충칭 시와 쓰촨성 서부 지역을 관할했던 군이다.',
   '기원전 316년, 진나라에서 파나라를 멸하고 그 자리에 처음으로 파군을 설치했다.',
   '치소는 강주현이라고도 하는 강현(현재의 충칭 시 북)이다.',
   '관할 구역은 대략 지금의 쓰촨성 랑중시 이동과 충칭시 푸링

In [14]:
valid_dataset[1]

(['바실레우스 또는 바실레프스는 그리스어로 군주를 의미하는 단어이다.',
  '이집트어 어근인 "파세르"(Paser)/"파시로"(Pasir)에서 유래했으며 초기에는 고관 또는 군대의 사령관을 나타냈다.',
  "고대 그리스어에서는 '바실레우스'로 발음되었지만 이후 '바실레프스'로 발음이 변화하고 이것이 현대 그리스어까지도 이어지고 있다.",
  '이 단어가 이집트어 "바시르"와 비슷한 점을 고려할 때 함족어에서 유래했을 가능성이 가장 크다.',
  '이 단어는 그리스어 명사와 관련이 있지만 미케네 그리스어에서는 "콰시레우"("qasirewu")로 나타난다.',
  '고대 시대에 바실레우스(βασιλεύς)는 과두 정치에서 같은 직무와 군대에서 다른 누구보다도 고귀한 혈통임을 드러냈다.',
  '그 시기에 더 높은 칭호로 왕을 의미하는 와낙스(wanax)가 존재했다.',
  '호메로스에서 왕은 신성한 존재로 여겨지며 하느님은 언제나 왕이시라고 하지만 바실레우스라는 언급은 없다.',
  '와낙스가 정계에서 물러난 후 미노스 왕 시대에 들어서면서 바실레우스는 점차 왕을 의미하게 되었다.',
  '나중에 페르시아의 왕은 전제 군주를 의미하는 유사한 축약어로 이 단어를 사용하기도 했으며 고대 후기부터 이 단어의 발음이 바실레우스에서 바실레프스로 점차 변화하기 시작했다.',
  "이 시기에 바실레프스는 라틴어로 로마 황제를 지칭하는 칭호 중 하나인 '아우구스투스'를 대신하게 되었다.",
  '동로마 제국은 7세기 초반부터 이 칭호를 사용했으며 페르시아 제국을 물리친 헤라클리우스 황제에 의해 널리 알려졌다.',
  '이 칭호의 전체 명칭은 황제와 동등함을 나타내는 "바실레프스 톤 로메온"(Βασιλες τν ωμαίων)이다.',
  '동로마는 1453년 오스만 제국에 의해 콘스탄티노플이 함락될 때까지 이 칭호를 계속 사용했다.',
  '이 칭호의 마지막 소유자는 콘스탄티누스 11세 팔라이올로고스였다.',
  '조이 여제와 같은 여왕들도 이 칭호를 사용한 적이 있다.',

In [15]:
test_dataset[1]

(['도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다.',
  '개인적인 측면과 사회적인 측면은 서로 밀접하게 연결되어 있기 때문이다.',
  '이는 인간이 본래 개인이면서 동시에 사회적 존재라는 사실에서 기인한다.',
  '그러나 도덕의 문제를 다룰 때는 개인적인 측면을 강조하는 경우도 있고, 사회적인 측면을 강조하는 경우도 존재한다.',
  '예를 들어, 인격의 자유나 양심의 가책, 선악의 판단과 같은 문제는 항상 개인이 중심이 되지만, 어른에 대한 예의, 길을 걷는 사람에 대한 친절, 빈곤한 사람에 대한 동정 등은 사회적 관계가 중심이 된다.'],
 -1)

## Define Model

In [16]:
class VectorMaxPool1d(nn.Module):
    def __init__(self, criterion='norm'):
        super().__init__()
        self.criterion = criterion

    def forward(self, x):
        if self.criterion == 'norm':
            norms = torch.norm(x, dim=-1)  # (batch_size, seq_len)
            max_indices = torch.argmax(norms, dim=-1)  # (batch_size,)

        elif self.criterion == 'max_value':
            max_values = torch.max(x, dim=-1)[0]  # (batch_size, seq_len)
            max_indices = torch.argmax(max_values, dim=-1)  # (batch_size,)

        elif self.criterion == 'sum':
            sums = torch.sum(x, dim=-1)  # (batch_size, seq_len)
            max_indices = torch.argmax(sums, dim=-1)  # (batch_size,)

        elif self.criterion == 'mean':
            means = torch.mean(x, dim=-1)  # (batch_size, seq_len)
            max_indices = torch.argmax(means, dim=-1)  # (batch_size,)

        else:
            raise ValueError(f"Unknown criterion: {self.criterion}")

        # extract the selected vectors
        batch_size = x.size(0)
        selected_vectors = x[torch.arange(batch_size), max_indices]

        return selected_vectors

In [17]:
class WeightedVectorMaxPool1d(nn.Module):
    def __init__(self, hidden_size, num_criteria=3):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_criteria = num_criteria

        self.criterion_weights = nn.Parameter(torch.ones(num_criteria))

    def forward(self, x):
        batch_size, seq_len, hidden_size = x.size()

        scores = []

        # first criterion: norm of each vector
        if self.num_criteria >= 1:
            norms = torch.norm(x, dim=-1)  # (batch_size, seq_len)
            scores.append(norms)

        # second criterion: max value in each vector
        if self.num_criteria >= 2:
            max_values = torch.max(x, dim=-1)[0]  # (batch_size, seq_len)
            scores.append(max_values)

        # third criterion: sum of values
        if self.num_criteria >= 3:
            sums = torch.sum(x, dim=-1)  # (batch_size, seq_len)
            scores.append(sums)

        # score calculation
        weighted_scores = torch.zeros_like(scores[0])
        weights = nn.functional.softmax(self.criterion_weights, dim=0)

        for i, score in enumerate(scores):
            weighted_scores += weights[i] * score

        # select the max vector based on the weighted scores
        max_indices = torch.argmax(weighted_scores, dim=-1)  # (batch_size,)
        selected_vectors = x[torch.arange(batch_size), max_indices]

        return selected_vectors

In [18]:
class BinaryFocalWithLogitsLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')

        pt = torch.sigmoid(inputs)
        pt = torch.where(targets == 1, pt, 1 - pt)

        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        focal_loss = alpha_t * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [19]:
base_model_id = "monologg/koelectra-small-v3-discriminator"
base_model_config = ElectraConfig.from_pretrained(base_model_id)

In [20]:
from typing import Optional

class ElectraForNaiveTextDetection(ElectraPreTrainedModel):
    def __init__(self, config: ElectraConfig = base_model_config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.electra = ElectraModel(config)
        self.classifier = nn.Sequential(
            VectorMaxPool1d('norm'),
            nn.Sequential(
                nn.Dropout(0.2),
                nn.Linear(self.config.hidden_size, self.config.hidden_size // 2),
                nn.LayerNorm(self.config.hidden_size // 2),
                nn.SiLU(),
                nn.Dropout(0.25),
                nn.Linear(self.config.hidden_size // 2, 1)
            )
        )

        # 3. Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids: list[torch.LongTensor],
        attention_mask: list[torch.Tensor],
        labels: Optional[torch.Tensor] = None,
        criterion = BinaryFocalWithLogitsLoss(),
        symmetric = nn.MSELoss(),
        alpha: float = 0.2,
    ) -> torch.Tensor:
        hidden_states = []
        for input_id, attention_mask in zip(input_ids, attention_mask):
            # forward pass by each paragraph
            hidden_state = self.electra(input_ids=input_id, attention_mask=attention_mask).last_hidden_state[:, 0, :]  # pooling by first token (CLS token)
            if torch.isnan(hidden_state).any():
                print("hidden_state", hidden_state)
            # [hidden_size, sentences] => [hidden_size]
            polled = self.classifier[0](hidden_state.unsqueeze(0))
            if torch.isnan(polled).any():
                print("polled", polled)
            hidden_states.append(polled)  # [1, hidden_size]

        hidden_states = torch.cat(hidden_states, dim=0)  # [batch_size, hidden_size]
        out = self.classifier[-1](hidden_states).squeeze(1)  # [batch_size]
        if torch.isnan(out).any():
            print("out", out)

        if labels is not None:
            out_symm = self.classifier[-1](hidden_states).squeeze(1)
            loss = (1-alpha) * criterion(out, labels) + alpha * symmetric(out, out_symm)
            return out, loss
        else:
            return out


try:  # For the case of reloading the model class
    for model in model_bag:
        model.__class__ = ElectraForNaiveTextDetection
except Exception:
    pass

In [21]:
model_bag = []
for bag in range(train_dataset.bagging_size):
    try:
        model = ElectraForNaiveTextDetection()

        from safetensors.torch import load_file
        state_dict = load_file(f"./models/{PROJECT_NAME}_last/model.safetensors")
        model.load_state_dict(state_dict)
    except Exception:
        model = ElectraForNaiveTextDetection.from_pretrained(base_model_id)

    model_bag.append(model)
    model = model.bfloat16()
    model.to(device)

model_bag[0]

ElectraForNaiveTextDetection(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(35000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear

In [22]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(device)

In [23]:
tokenize("Hello, my dog is cute")

{'input_ids': tensor([[    2, 19831, 18268,  4008,    16, 19675, 23215,  4060, 12557,    71,
         24363,     3]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

## Train and Evaluate

### Utils

In [24]:
def to_label(scores, threshold=0.5):
    return [1 if score > threshold else 0 for score in scores]

def calc_score(ls, lb, pd):
    def do_by_bag(ls, lb, pd):
        pd_discrete = to_label(pd)
        if len(ls) == 0:
            lss = 0.0
        else:
            lss = sum(ls) / len(ls)
        if len(lb) == 0:
            acc, f1, rocauc = 0.0, 0.0, 0.0
        else:
            acc = accuracy_score(lb, pd_discrete)
            f1 = f1_score(lb, pd_discrete)
            rocauc = roc_auc_score(lb, pd)
        return lss, acc, f1, rocauc

    lss, acc, f1, rocauc = zip(*[do_by_bag(ls[i], lb[i], pd[i]) for i in range(len(ls))])
    lss, acc, f1, rocauc = (" ".join("{:.4f}".format(i) for i in x) for x in (lss, acc, f1, rocauc))
    return f"Loss: {lss}, ACC: {acc}, F1: {f1}, ROCAUC: {rocauc}"

In [25]:
BATCH_SIZE = 8, 16, 8
REAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = REAL_BATCH_SIZE // BATCH_SIZE[0]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True, collate_fn=lambda batch: [tuple(zip(*x)) for x in tuple(zip(*batch))])
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

In [26]:
EPOCHS = 5
START_EPOCH = 0
LEARNING_RATE = 1e-4
BACKBONE_LEARNING_RATE = 2e-5

optimizer = [optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE) for model in model_bag]
backbone_optimizer = [optim.AdamW(model.electra.parameters(), lr=BACKBONE_LEARNING_RATE) for model in model_bag]

In [27]:
next(iter(train_loader))[0]

((['이란인의 형성과 메디아 시대.',
   '이란 고원에는 아주 오래전부터 사람들이 살았습니다.',
   '아리아인들은 여러 부족으로 나뉘어 흩어지고 모였다가 하는 과정을 거쳤는데 그중 스키타이족 메디아족 그리고 페르시아인들을 포함한 이란족이 모두 아리아인의 후손입니다.',
   '초창기 이란의 아리아인 즉 이란족은 메소포타미아의 수메르나 바빌로니아에 맞서 싸우는 용병이었습니다.',
   "그들은 고원을 장악하고 '이란' 아리아인의 땅을 건설했습니다.",
   '기원전 7세기경 이란인의 한 갈래인 메디아인들이 아시리아로부터 독립하여 남부 이란과 소아시아에 걸쳐 메디아 왕국을 세웠습니다.',
   '기원전 708년부터 기원전 550년까지 이어진 메디아 왕국은 이란인이 세운 최초의 왕조였죠.',
   '하지만 중앙 집권 국가를 이루지 못하고 부족 연합체로 남았습니다.',
   '피란샤르시는 이란에서 가장 오래된 문명이며 8000년의 역사를 가지고 있다.',
   '기원전 621년 메디아 왕국 아스티아게스 왕 시절 아리아인들은 메소포타미아를 손에 넣었습니다.',
   '아스티아게스 왕은 바빌론과 손을 잡고 아시리아를 무너뜨렸고 메소포타미아 북부를 장악했죠.',
   '메디아는 티그리스 유프라테스 강 유역의 비옥한 땅 오늘날 이라크를 차지하려 신 바빌로니아 왕국과 싸웠지만 결국 패배했습니다.',
   '바빌로니아의 나보니두스 왕은 이란 남부 아케메네스 왕조 (기원전 550년 ~ 기원전 330년) 와 동맹을 맺어 메디아를 정벌하였고, 아케메네스는 아스티아게스의 외손자인 키루스 2세(Cyrus the Great)가 연 왕조다.',
   '아스티아게스는 아시리아를 무너뜨리기 위해 바빌론과 손잡았다가 훗날 바빌론에 망했고, 키루스는 바빌론과 연합해 메디아를 무너뜨리더니 급기야는 바빌론에 칼을 돌렸다.',
   '키루스는 주변 부족 국가들을 통합해 동으로는 소아시아와 아르메니아, 서로는 힌두쿠시까지 세력을 확장했고 기원전 539년 바빌로니아를 정벌한다.',
   '한때의 

In [28]:
next(iter(valid_loader))

((['샤쥔춘(沙俊春, 1984년 3월 17일~)은 중국 출신의 스타크래프트 프로게이머였습니다.',
   'SK텔레콤 T1 팀 소속으로 프로토스를 주로 사용했죠. 2004년 루오시안과 같이 SK텔레콤 T1에 입단하였다. 2007년 5월 한국에서의 선수 생활을 끝내고 중국으로 돌아갔습니다.',
   '그렇게 그의 국내 무대는 막을 내렸습니다.',
   'WCG 국가대표로 출전해서 WCG 2007 그랜드 파이널 결승에선 대한민국의 송병구에게 패했지만 은메달을 따냈다. 2009년 스웨덴과 독일의 이스포츠팀인 SK 게이밍에 입단하였다. 2009년에는 중국에서 프로게임팀 소속으로 선수이자 감독을 맡았습니다. 2010년 게임단이 해체된 후 은퇴했고요 지금은 개인 사업을 하고 있습니다.',
   "WCG2007 그랜드파이널 8강에서 한국의 프로게이머 마재윤을 2:1로 격파하면서 '샤본좌, 대륙의혁명가' 등의 별명을 얻었다.",
   '반면 마재윤은 큰 비난을 받으며 마완용이라는 불명예스러운 별명까지 얻었다.',
   '한국 선수들과는 늘 끈질긴 악연이었죠.',
   '쉽게 이길 수 없었던 상대였습니다.',
   '그들의 힘과 투지는 정말 대단했어요.',
   '한국 선수와의 경기는 늘 긴장의 연속이었고 승부를 예측할 수 없었습니다.',
   '샤쥔춘은 외국 프로게이머로서는 한국 프로게이머를 상당수 잡은 몇 안되는 프로게이머로도 유명하다.',
   '이 중에서 그간 잡은 프로게이머 목록이 최연성 마재윤 김정우 등 국내 유명 프로게이머 상당수일 정도.',
   '최근에는 IEF 2009에서 김정우에게 전진 2게이트를 선보이면서 승리를 챙기기도 했으며, 이 때 팬들은 다시 그를 보고 샤본좌 라고 연호했고 김정우는 매완용, 매데요 시 라는 불명예스러운 별명을 얻고야 만다.',
   '김택용과의 끈질긴 인연.',
   '샤쥔춘은 마재윤과 김정우를 모두 꺾고 최연성까지 잡았지만 김택용만은 넘지 못했습니다.',
   'IEF 2008에서는 3대 0 완패를 당했고 IEF 2009에서도 

In [29]:
next(iter(test_loader))

((['도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다.',
   '그러므로 도덕은 어디까지나 정신의 문제이고, 각자의 마음씨에 달려있는 일이다.',
   '여기에서 도덕의 문제는 철학적 이론으로 발전하였으며, 고상하고 심원한 이론 체계에 기울어지는 경향이 많았다.',
   '이러한 경향으로 인해 도덕은 학식이 높은 특수한 사람만이 닦을 수 있는 것으로 여겨지며, 일반 사람은 도저히 지킬 수 없는 것처럼 오해받는 경우가 많았다.',
   '그러나 도덕의 본질은 결코 이론에 있는 것이 아니라 실천에 있다.',
   '이론이 필요하다면 그것은 오직 실천을 위한 도구로서만 의미를 가진다.',
   '아무리 고상한 이론이라도 실천이 없다면 그 이론은 단순한 관념의 유희에 불과하다.'],
  ['도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다.',
   '개인적인 측면과 사회적인 측면은 서로 밀접하게 연결되어 있기 때문이다.',
   '이는 인간이 본래 개인이면서 동시에 사회적 존재라는 사실에서 기인한다.',
   '그러나 도덕의 문제를 다룰 때는 개인적인 측면을 강조하는 경우도 있고, 사회적인 측면을 강조하는 경우도 존재한다.',
   '예를 들어, 인격의 자유나 양심의 가책, 선악의 판단과 같은 문제는 항상 개인이 중심이 되지만, 어른에 대한 예의, 길을 걷는 사람에 대한 친절, 빈곤한 사람에 대한 동정 등은 사회적 관계가 중심이 된다.'],
  ['여기에 이른바 공중도덕은 실천적, 사회적 도덕의 한 부문이다.',
   '즉, 공중 도덕이라 함은 여러 사람이 다 함께 편리하고 유익하며 정결하고 질서있게 공동생활을 하기 위해 지켜야 할 도덕을 말한다.',
   '여러 사람의 공동생활에 지장을 초래하거나 방해를 일으킬 일에 대해서는 법률로써 이를 금지하는 경우도 있으나, 이를 낱낱이 모두 법률로 규정하기 어려울 뿐만 아니라, 설령 법률로써 금지하여 있다 할지라도 도덕적 노력 없이는 사회의 안녕과 질서를 얻기 어려울 것이다.',
   '공중

### Training Loop

In [31]:
with (
    tqdm(range(START_EPOCH, START_EPOCH+EPOCHS), desc="[Running Epochs]") as epochs,
    tqdm(range(len(train_dataset)//REAL_BATCH_SIZE), desc="[Training]") as train_progress,
    tqdm(range(len(valid_dataset)//BATCH_SIZE[1]), desc="[Validating]") as valid_progress
):
    for epoch in epochs:
        train_progress.reset()
        train_loss, train_preds, train_labels, train_accums = ([[] for _ in range(train_dataset.bagging_size)] for _ in range(4))

        # Model Preparation
        for model in model_bag:
            if epoch < 2:
                for param in model.electra.parameters():
                    param.requires_grad = False
            else:
                for param in model.electra.parameters():
                    param.requires_grad = True

        # Train
        for step, bagged_data in enumerate(train_loader):
            for bag, model, (texts, labels) in zip(range(1), model_bag, bagged_data):
                torch.cuda.empty_cache(); gc.collect();
                model.train()

                labels = torch.tensor(labels).to(device)
                input_ids = []
                attention_masks = []
                for tokenized, ori in zip(map(tokenize, texts), texts):
                    ids = tokenized['input_ids']
                    if len(tokenized['input_ids'][0]) > 512:
                        print(ids, file=sys.stderr)
                        #raise ValueError("Input length exceeds 512 tokens.", f"Input length: {len(ids[0])}", ori)
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                try:
                    logits, loss = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels.float())
                    scores = torch.sigmoid(logits).tolist()
                    train_accums[bag].append(loss)
                    train_preds[bag].extend(scores)
                    train_labels[bag].extend(labels.tolist())
                    train_loss[bag].append(loss.item())

                    if (step+1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step+1) == len(train_loader):
                        total_loss = torch.mean(torch.stack(train_accums[bag]))
                        total_loss.backward()
                        optimizer[bag].step()
                        backbone_optimizer[bag].step()
                        optimizer[bag].zero_grad()
                        backbone_optimizer[bag].zero_grad()
                        train_accums[bag] = []

                        train_progress.update(1)
                        train_progress.set_description(f"[Training] " + calc_score(train_loss, train_labels, train_preds))
                except Exception as e:
                    if "CUDA out of memory" in str(e):
                        print(e, file=sys.stderr)
                    else:
                        pass#raise e

        # Validate
        valid_loss, valid_preds, valid_labels = ([[] for _ in range(train_dataset.bagging_size + 1)] for _ in range(3))
        valid_progress.reset()
        for texts, labels in valid_loader:
            torch.cuda.empty_cache(); gc.collect();
            labels = torch.tensor(labels).to(device)
            input_ids = []
            attention_masks = []
            for tokenized in [tokenize(t) for t in texts]:
                input_ids.append(tokenized['input_ids'].to(device))
                attention_masks.append(tokenized['attention_mask'].to(device))

            try:
                bagged_preds = [[] for _ in range(train_dataset.bagging_size)]
                for bag, model in zip(range(1), model_bag):
                    model.eval()
                    with torch.no_grad():
                        logits, loss = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels.float())
                        scores = torch.sigmoid(logits).tolist()

                        valid_preds[bag].extend(scores)
                        bagged_preds[bag].extend(scores.item())
                        valid_labels[bag].extend(labels.tolist())
                        valid_loss[bag].append(loss.item())
                valid_preds[-1].extend(sum(p) / len(p) for p in zip(*bagged_preds))

                valid_progress.set_description(f"[Validating] " + calc_score(valid_loss, valid_labels*len(model_bag), valid_preds))
            except Exception as e:
                if "CUDA out of memory" in str(e):
                    print(e, file=sys.stderr)
                else:
                    pass#raise e

            valid_progress.update(1)

        model.save_pretrained(f"./models/{PROJECT_NAME}_{epoch}")
        #model.save_pretrained(f"./models/{PROJECT_NAME}_last")
        START_EPOCH = epoch

[Running Epochs]:   0%|          | 0/5 [00:00<?, ?it/s]

[Training]:   0%|          | 0/1349 [00:00<?, ?it/s]

[Validating]:   0%|          | 0/100 [00:00<?, ?it/s]

tensor([[    2,  6340, 10896,  ...,     0,     0,     0],
        [    2,  2010,  4301,  ...,  4291,  7197,     3]], device='cuda:0')
tensor([[    2,  2828,  4343,  ...,     0,     0,     0],
        [    2, 30910, 22460,  ...,     0,     0,     0],
        [    2,  6339, 25082,  ...,     0,     0,     0],
        ...,
        [    2,  6599,  4113,  ...,     0,     0,     0],
        [    2, 11028,  3071,  ...,     0,     0,     0],
        [    2,  6265, 18053,  ...,    16,  6244,     3]], device='cuda:0')
tensor([[    2,  2800,  4042,  ...,     0,     0,     0],
        [    2,  2800,  4042,  ...,     0,     0,     0],
        [    2, 14035,  4034,  ...,     0,     0,     0],
        ...,
        [    2, 14083,  4258,  ...,     0,     0,     0],
        [    2,  6618,  6320,  ...,  4176,    18,     3],
        [    2,  6244, 10749,  ...,     0,     0,     0]], device='cuda:0')
tensor([[    2, 15960,  4112,  ...,     0,     0,     0],
        [    2, 25033,  4239,  ...,     0,     0, 

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Model Saving
model.save_pretrained(f"./models/{PROJECT_NAME}_last")

### Final Output

In [ ]:
results = []
with tqdm(test_dataset_bundled, desc="[Testing]") as progress:
    humans, ais = 0, 0
    model.eval()
    for texts in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in texts]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                scores = torch.sigmoid(logits)
                results.extend(scores.tolist())
                for preds in to_label(scores):
                    if preds == 0:
                        humans += 1
                    else:
                        ais += 1
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Testing] Human: {humans/len(test_dataset):.2%}, Ai: {ais/len(test_dataset):.2%}")

len(results) == len(test_dataset)

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x="generated", kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv(f"./data/submission_{PROJECT_NAME[2:]}.csv", index=False, encoding='utf-8-sig')